# 08. 가로수 데이터 추가

## 1) import

In [26]:
from pathlib import Path
import math, random, datetime, pytz
import numpy as np
import pandas as pd
from pyproj import Transformer

# 재사용 유틸 (너의 04/05/07에서 가져온 것)
from pysolar.solar import get_altitude, get_azimuth

## 2) 좌표계 변환(WGS84 <-> UTM 52N)

In [27]:
EPSG_WGS84 = 4326 
EPSG_UTM52 = 32652

In [28]:
tf_wgs84_to_utm = Transformer.from_crs(EPSG_WGS84, EPSG_UTM52, always_xy=True)  # (lon,lat)->(E,N)
tf_utm_to_wgs84 = Transformer.from_crs(EPSG_UTM52, EPSG_WGS84, always_xy=True)  # (E,N)->(lon,lat)

In [29]:
def ll_to_utm(lon, lat):
    E, N = tf_wgs84_to_utm.transform(lon, lat)
    return float(E), float(N)

def utm_to_ll(E, N):
    lon, lat = tf_utm_to_wgs84.transform(E, N)
    return float(lon), float(lat)

## 3) 데이터 로드 & 셀(정사각형) 코너 계산

In [30]:
TREES_CSV = Path(r"..\data\경북대_가로수_위경도.csv")

In [31]:
df_grid = pd.read_csv(TREES_CSV)

df_grid["NUMPOINTS"] = pd.to_numeric(df_grid["NUMPOINTS"], errors="coerce").fillna(0).astype(int)
df_grid["grid_size_m"] = pd.to_numeric(df_grid["grid_size_m"], errors="coerce")

In [33]:
df_grid = df_grid[df_grid["grid_size_m"] == 100].copy()
df_grid = df_grid[df_grid["NUMPOINTS"] > 0].reset_index(drop=True)

In [34]:
df_grid.head()

,id,NUMPOINTS,lon,lat,grid_size_m
0,102274,6,128.601156,35.891303,100
1,102275,23,128.601144,35.890402,100
2,102276,24,128.601131,35.889500,100
3,102277,10,128.601119,35.888599,100
4,102279,2,128.601094,35.886796,100


In [40]:
E_list, N_list = [], []
for lon, lat in zip(df_grid["lon"], df_grid["lat"]):
    E, N = ll_to_utm(lon, lat)
    E_list.append(E); N_list.append(N)
df_grid["E_center"] = E_list
df_grid["N_center"] = N_list

In [41]:
print("100m cells:", len(df_grid))
df_grid.head()

100m cells: 124


,id,NUMPOINTS,lon,lat,grid_size_m,E_center,N_center,corners_EN
0,102274,6,128.601156,35.891303,100,464003.809602,3.971966e+06,"[(463953.80960239464, 3971915.968985496), (464..."
1,102275,23,128.601144,35.890402,100,464002.274948,3.971866e+06,"[(463952.27494769474, 3971815.991339868), (464..."
2,102276,24,128.601131,35.889500,100,464000.740326,3.971766e+06,"[(463950.7403263574, 3971716.0136935916), (464..."
3,102277,10,128.601119,35.888599,100,463999.205738,3.971666e+06,"[(463949.20573838515, 3971616.0360466707), (46..."
4,102279,2,128.601094,35.886796,100,463996.136663,3.971466e+06,"[(463946.13666252367, 3971416.0807508905), (46..."


## 4) 포아송 디스크 샘플링(정사각형 내부에 정확히 NUMPOINTS개)

항상 100×100m 정사각형 내부에서만 수행.  
최소 간격 r은 자동 추정 → 부족하면 r을 줄여 재시도 → 최종 n개 맞춤.

In [62]:
def poisson_in_square(center_E, center_N, size_m, n, seed=42, max_tries=5):
    """
    100m 정사각형 내부 포아송 디스크 샘플링으로 '최소 n개'를 만들고,
    정확히 n개가 되도록 랜덤 서브샘플. 실패 시 균일 난수 fallback.
    """
    rng = random.Random(seed)
    A = size_m * size_m
    alpha = 0.85
    r = alpha * math.sqrt(A / (max(n,1) * math.pi))  # 초기 최소간격 추정(셀/밀도 기반)
    h = 0.5 * size_m

    for _ in range(max_tries):
        pts = []
        active = []
        # 첫 점
        x0 = rng.uniform(-h, h); y0 = rng.uniform(-h, h)
        pts.append((x0, y0)); active.append((x0, y0))

        k = 30
        r2 = r*r

        def in_square(x, y): return (-h <= x <= h) and (-h <= y <= h)
        def far_enough(x, y):
            for (px, py) in pts:
                dx = x - px; dy = y - py
                if dx*dx + dy*dy < r2:
                    return False
            return True

        while active and len(pts) < n:
            ax, ay = active[rng.randrange(len(active))]
            found = False
            for _ in range(k):
                rho = rng.uniform(r, 2*r); th = rng.uniform(0, 2*math.pi)
                nx = ax + rho * math.cos(th); ny = ay + rho * math.sin(th)
                if in_square(nx, ny) and far_enough(nx, ny):
                    pts.append((nx, ny)); active.append((nx, ny))
                    found = True
                    if len(pts) >= n: break
            if not found:
                active.remove((ax, ay))

        if len(pts) >= n:
            rng.shuffle(pts); pts = pts[:n]
            return [(center_E + x, center_N + y) for (x, y) in pts]

        r *= 0.85  # 실패시 r 줄여서 재시도

    # fallback: 최소간격 고려 없이 균일 난수로 n개
    pts = [(rng.uniform(-h, h), rng.uniform(-h, h)) for _ in range(n)]
    return [(center_E + x, center_N + y) for (x, y) in pts]


## 5) 나무 파라미터 샘플링(높이/수관/줄기)

플라타너스 기준
* https://namu.wiki/w/%EB%B2%84%EC%A6%98%EB%82%98%EB%AC%B4  
* https://treeworld.co.kr/a01_01_02/29800 

In [64]:
def sample_tree_params(n, rng=np.random.default_rng(0),
                       height_mean=8.0, height_std=2.0,
                       crown_r_min=2.0, crown_r_max=4.0,
                       trunk_r_min=0.10, trunk_r_max=0.25):
    heights = np.clip(rng.normal(height_mean, height_std, size=n), 3.0, 20.0)
    crown_r = rng.uniform(crown_r_min, crown_r_max, size=n)
    trunk_r = rng.uniform(trunk_r_min, trunk_r_max, size=n)
    return heights, crown_r, trunk_r

## 6) 100m 셀마다 나무 샘플링 → 개별 나무 표 (UTM/위경도)

In [46]:
trees = []
seed_base = 12345

In [66]:
for i, row in df_grid.iterrows():
    n = int(row["NUMPOINTS"])
    E0, N0 = row["E_center"], row["N_center"]
    pts_EN = poisson_in_square(E0, N0, 100, n, seed=seed_base + i)

    heights, crown_r, trunk_r = sample_tree_params(
        n, rng=np.random.default_rng(seed_base + i),
        height_mean=8.0, height_std=2.0,
        crown_r_min=2.0, crown_r_max=4.0,
        trunk_r_min=0.10, trunk_r_max=0.25
    )

    for j, (E, N) in enumerate(pts_EN):
        lon, lat = utm_to_ll(E, N)
        trees.append({
            "grid_id": row["id"],
            "E": E, "N": N,
            "lon": lon, "lat": lat,
            "height": float(heights[j]),
            "crown_r": float(crown_r[j]),
            "trunk_r": float(trunk_r[j])
        })

In [49]:
df_trees = pd.DataFrame(trees)
print("생성된 가로수 개수:", len(df_trees))
df_trees.head()

생성된 가로수 개수: 1745


,grid_id,E,N,lon,lat,height,crown_r,trunk_r
0,102274,464042.898370,3.971920e+06,128.601591,35.890889,5.152350,3.196618,0.200086
1,102274,464016.309511,3.971928e+06,128.601297,35.890962,10.527457,2.373468,0.114385
2,102274,463979.742323,3.971949e+06,128.600890,35.891152,6.258677,3.345512,0.166276
3,102274,463995.471590,3.971917e+06,128.601066,35.890861,7.481654,3.883606,0.232972
4,102274,464007.673702,3.971946e+06,128.601200,35.891124,7.849313,2.496491,0.204618


## 7) ENU 변환(그래프와 동일 기준) + 가로수 그림자 직사각형 생성

In [68]:
ARTIFACTS_DIR = Path("../artifacts")

In [69]:
# 그래프/건물에서 쓰던 기준점 로드(평균 위경도 예시)
with open(ARTIFACTS_DIR / "graph_xy.pkl", "rb") as f:
    import pickle
    G = pickle.load(f)
lat0 = float(np.mean([G.nodes[n]["lat"] for n in G.nodes]))
lng0 = float(np.mean([G.nodes[n]["lng"] for n in G.nodes]))
R_earth = 6371000.0

In [70]:
def ll_to_xy(lat, lng, lat0=lat0, lng0=lng0):
    dlat = math.radians(lat - lat0)
    dlng = math.radians(lng - lng0)
    x = R_earth * dlng * math.cos(math.radians(lat0))
    y = R_earth * dlat
    return x, y

In [71]:
# ShadowRect (건물과 동일 구조)
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple

In [73]:
@dataclass
class ShadowRect:
    cx: float; cy: float
    length: float; width: float
    dx: float; dy: float
    meta: Dict[str, Any]
    def endpoints(self):
        hx = 0.5*self.length*self.dx; hy = 0.5*self.length*self.dy
        return (self.cx - hx, self.cy - hy), (self.cx + hx, self.cy + hy)
    def corners(self):
        hx = 0.5*self.length*self.dx; hy = 0.5*self.length*self.dy
        nx, ny = -self.dy, self.dx; half_w = 0.5*self.width
        c1x, c1y = self.cx - hx, self.cy - hy
        c2x, c2y = self.cx + hx, self.cy + hy
        return [(c1x - half_w*nx, c1y - half_w*ny),
                (c1x + half_w*nx, c1y + half_w*ny),
                (c2x + half_w*nx, c2y + half_w*ny),
                (c2x - half_w*nx, c2y - half_w*ny)]

In [74]:
KST = pytz.timezone("Asia/Seoul")

In [76]:
def build_tree_shadow_rects(df_trees: pd.DataFrame, dt_local: datetime.datetime):
    dt_local = dt_local if dt_local.tzinfo else KST.localize(dt_local)
    dt_utc = dt_local.astimezone(pytz.UTC)
    rects = []

    for r in df_trees.itertuples(index=False):
        alt = get_altitude(r.lat, r.lon, dt_utc)
        if alt <= 0:     # 밤/지평선 아래면 스킵
            continue
        az  = get_azimuth(r.lat, r.lon, dt_utc)
        L   = float(r.height) / math.tan(math.radians(alt))

        # 태양 반대 방향 단위벡터 d
        th = math.radians(az); ux, uy = math.sin(th), math.cos(th)
        dx, dy = -ux, -uy
        nrm = math.hypot(dx,dy); dx, dy = dx/nrm, dy/nrm

        # 나무 중심 ENU
        x0, y0 = ll_to_xy(r.lat, r.lon, lat0, lng0)
        cx = x0 + 0.5 * L * dx
        cy = y0 + 0.5 * L * dy
        width = 2.0 * float(r.crown_r)  # 수관 반지름 기반

        rects.append(ShadowRect(cx=cx, cy=cy, length=L, width=width, dx=dx, dy=dy,
                                meta={"type":"tree", "height":float(r.height),
                                      "crown_r":float(r.crown_r)}))
    return rects

In [77]:
# 테스트: 오늘 13시
now = datetime.datetime.now(KST).replace(minute=0, second=0, microsecond=0)
test_time = now.replace(hour=13)
tree_rects = build_tree_shadow_rects(df_trees, test_time)
len(tree_rects), tree_rects[0].corners() if tree_rects else None

(1745,
 [(-896.3747271648538, 147.78397987776745),
  (-902.4702277133401, 149.71227627928303),
  (-901.3516782689032, 153.24810134575867),
  (-895.2561777204169, 151.3198049442431)])

## 8) 빌딩 + 가로수 그림자 합치기 → 간선 그늘 재계산

In [78]:
import pickle, pandas as pd
from pathlib import Path

In [80]:
# 이미 저장해둔 직사각형이 있으면 로드
with open(ARTIFACTS_DIR / "shadow_rects.pkl", "rb") as f:
    rects_buildings = pickle.load(f)

In [81]:
import numpy as np

In [82]:
def segment_rect_interval(p0, p1, rect):
    d = np.array([rect.dx, rect.dy], dtype=float)
    n = np.array([-rect.dy, rect.dx], dtype=float)
    c = np.array([rect.cx, rect.cy], dtype=float)
    p0 = np.array(p0, dtype=float) - c
    p1 = np.array(p1, dtype=float) - c
    dp = p1 - p0
    u0, v0 = float(np.dot(p0, d)), float(np.dot(p0, n))
    du, dv = float(np.dot(dp, d)), float(np.dot(dp, n))
    umin, umax = -0.5*rect.length, 0.5*rect.length
    vmin, vmax = -0.5*rect.width,  0.5*rect.width
    t0, t1 = 0.0, 1.0
    def clip(p, q, t0, t1):
        if p == 0: return (t0, t1) if q <= 0 else (None, None)
        r = q/p
        if p > 0: t0 = max(t0, r)
        else:     t1 = min(t1, r)
        return (t0, t1) if t0 <= t1 else (None, None)
    t = clip( du, umin - u0, t0, t1);  t0,t1 = t if t!=(None,None) else (None,None)
    if t0 is None: return []
    t = clip(-du, umax - u0, t0, t1);  t0,t1 = t if t!=(None,None) else (None,None)
    if t0 is None: return []
    t = clip( dv, vmin - v0, t0, t1);  t0,t1 = t if t!=(None,None) else (None,None)
    if t0 is None: return []
    t = clip(-dv, vmax - v0, t0, t1);  t0,t1 = t if t!=(None,None) else (None,None)
    if t0 is None: return []
    if t0 <= t1 and t1 >= 0 and t0 <= 1:
        return [(max(0.0, t0), min(1.0, t1))]
    return []

In [83]:
def merge_intervals(intervals, eps=1e-12):
    if not intervals: return []
    intervals = sorted(intervals, key=lambda x: x[0])
    merged = [list(intervals[0])]
    for a,b in intervals[1:]:
        if a <= merged[-1][1] + eps:
            merged[-1][1] = max(merged[-1][1], b)
        else:
            merged.append([a,b])
    return [(a,b) for a,b in merged]

In [84]:
def compute_unique_shade_length_for_edge(p0, p1, rects):
    ex = p1[0]-p0[0]; ey = p1[1]-p0[1]
    L = math.hypot(ex,ey)
    if L == 0: return 0.0
    ebox = (min(p0[0],p1[0]), min(p0[1],p1[1]), max(p0[0],p1[0]), max(p0[1],p1[1]))
    intervals = []
    for r in rects:
        xs = [p[0] for p in r.corners()]; ys = [p[1] for p in r.corners()]
        rb = (min(xs), min(ys), max(xs), max(ys))
        if not (ebox[2] >= rb[0] and rb[2] >= ebox[0] and ebox[3] >= rb[1] and rb[3] >= ebox[1]):
            continue
        intervals += segment_rect_interval(p0, p1, r)
    merged = merge_intervals(intervals)
    return sum((b-a) for a,b in merged) * L

In [85]:
# 빌딩 + 가로수 그림자 합치기
rects_all = rects_buildings + tree_rects

In [86]:
# 간선 그늘 길이 갱신
for u, v, d in G.edges(data=True):
    p0 = (G.nodes[u]["x"], G.nodes[u]["y"])
    p1 = (G.nodes[v]["x"], G.nodes[v]["y"])
    d["shaded_len_m"] = compute_unique_shade_length_for_edge(p0, p1, rects_all)


In [87]:
# 통계 확인
shades = [d.get("shaded_len_m", 0.0) for _,_,d in G.edges(data=True)]
print(pd.Series(shades).describe())

count    440.000000
mean       1.813192
std        8.971435
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max       66.712055
dtype: float64


## 9) 저장

In [88]:
with open(ARTIFACTS_DIR / "graph_with_tree_shade.pkl", "wb") as f:
    pickle.dump(G, f)

In [89]:
# 필요시 가로수 직사각형도 저장(클래스 경로 문제 피하려면 dict로 직렬화 권장)
def rect_to_dict(r):
    return {"cx":r.cx, "cy":r.cy, "length":r.length, "width":r.width, "dx":r.dx, "dy":r.dy, "meta":r.meta}


In [90]:
import pickle
with open(ARTIFACTS_DIR / "tree_shadow_rects.pkl", "wb") as f:
    pickle.dump([rect_to_dict(r) for r in tree_rects], f)

## 10) 시각화 테스트

In [91]:
import math
import folium
from folium.plugins import MarkerCluster

In [92]:
R_earth = 6371000.0
def xy_to_ll(x, y, lat0=lat0, lng0=lng0):
    dlat = y / R_earth
    dlng = x / (R_earth * math.cos(math.radians(lat0)))
    return (math.degrees(dlat) + lat0, math.degrees(dlng) + lng0)

In [93]:
m = folium.Map(location=[lat0, lng0], zoom_start=16, tiles=None)
folium.TileLayer("OpenStreetMap", name="OSM (Color)").add_to(m)
folium.TileLayer("cartodbpositron", name="CartoDB Positron (Light)").add_to(m)
folium.TileLayer("cartodbdark_matter", name="CartoDB DarkMatter (Dark)").add_to(m)

도로 네트워크 얇게 깔기

In [94]:
roads_fg = folium.FeatureGroup(name="Road network", show=True)
for u, v in G.edges:
    roads_fg.add_child(folium.PolyLine(
        [(G.nodes[u]["lat"], G.nodes[u]["lng"]),
         (G.nodes[v]["lat"], G.nodes[v]["lng"])],
        weight=2, opacity=0.30, color="#3388ff"
    ))
roads_fg.add_to(m)

건물 그림자 레이어 (많으면 무거울 수 있음)

In [ ]:
show_building_shadows = True  # 필요 없으면 False
if show_building_shadows and 'rects_buildings' in globals() and rects_buildings:
    b_fg = folium.FeatureGroup(name="Building Shadows", show=False)
    for r in rects_buildings:
        pts_ll = [xy_to_ll(x, y) for (x, y) in r.corners()]
        folium.Polygon(pts_ll, color="#000000", weight=1,
                       fill=True, fill_opacity=0.18).add_to(b_fg)
    b_fg.add_to(m)

가로수 그림자 레이어

In [95]:
tree_shadow_fg = folium.FeatureGroup(name="Tree Shadows", show=True)

In [96]:
for r in tree_rects:
    pts_ll = [xy_to_ll(x, y) for (x, y) in r.corners()]
    # 수관 반지름/높이 정보를 팝업으로
    crown = r.meta.get("crown_r", None)
    h = r.meta.get("height", None)
    popup = None
    if crown is not None and h is not None:
        popup = folium.Popup(f"Tree shadow<br>height≈{h:.1f} m<br>crown_r≈{crown:.1f} m", max_width=220)
    folium.Polygon(pts_ll,
                   color="#2ca25f", weight=1.0,
                   fill=True, fill_opacity=0.22,
                   popup=popup).add_to(tree_shadow_fg)
    
tree_shadow_fg.add_to(m)

가로수 점(좌표) 레이어

In [97]:
tree_point_fg = folium.FeatureGroup(name="Tree Points", show=True)
cluster = MarkerCluster(name="Tree Points (cluster)").add_to(tree_point_fg)

In [98]:
MAX_POINTS = 2000  # 필요시 변경
df_vis = df_trees if len(df_trees) <= MAX_POINTS else df_trees.sample(MAX_POINTS, random_state=0)

In [99]:
for r in df_vis.itertuples(index=False):
    tip = f"Tree point (≈ sampled)<br>h={getattr(r,'height', float('nan')):.1f} m, crown_r={getattr(r,'crown_r', float('nan')):.1f} m"
    folium.CircleMarker(
        location=(r.lat, r.lon),
        radius=2.5, color="#1b7837", fill=True, fill_opacity=0.9,
        tooltip=tip
    ).add_to(cluster)

In [100]:
tree_point_fg.add_to(m)

In [104]:
m.save("../artifacts/trees_shadow_map.html")